In [1]:
# Install OpenNMT-py 3.x
!pip3 install OpenNMT-py

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 262.8/262.8 KB 13.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.0/17.0 MB 125.3 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.1/104.1 KB 67.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.2/56.2 KB 40.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 190.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 755.5/755.5 MB 4.3 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 110.7/110.7 KB 70.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 103.0/103.0 KB 67.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 29.2/29.2 MB 72.4 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 189.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 38.4/38.4 MB 61.2 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.4/4.4 MB 189.7 MB/s eta

## Config for BERT-WSD

In [16]:
# Create the YAML configuration file
# On a regular machine, you can create it manually or with nano
# Note here we are using some smaller values because the dataset is small
# For larger datasets, consider increasing: train_steps, valid_steps, warmup_steps, save_checkpoint_steps, keep_checkpoint

config = '''# config.yaml


## Where the samples will be written
save_data: run

# Training files
data:
    corpus_1:
        path_src: en-zh.en-filtered-wsd-processed-salient.en.subword.train
        path_tgt: en-zh.zh-filtered-wsd.zh.subword.train
        transforms: [filtertoolong]
    valid:
        path_src: en-zh.en-filtered-wsd-processed-salient.en.subword.dev
        path_tgt: en-zh.zh-filtered-wsd.zh.subword.dev
        transforms: [filtertoolong]

# Vocabulary files, generated by onmt_build_vocab
src_vocab: run/source.vocab
tgt_vocab: run/target.vocab

# Vocabulary size - should be the same as in sentence piece
src_vocab_size: 10000
tgt_vocab_size: 10000

# Filter out source/target longer than n if [filtertoolong] enabled
src_seq_length: 512
src_seq_length: 512

# Tokenization options
src_subword_model: source.model
tgt_subword_model: target.model

# Where to save the log file and the output models/checkpoints
log_file: train.log
save_model: models/model.fren

# Stop training if it does not improve after n validations
early_stopping: 4

# Default: 5000 - Save a model checkpoint for each n
save_checkpoint_steps: 2000

# To save space, limit checkpoints to last n
# keep_checkpoint: 3

seed: 3435

# Default: 100000 - Train the model to max n steps 
# Increase to 200000 or more for large datasets
# For fine-tuning, add up the required steps to the original steps
train_steps: 10000

# Default: 10000 - Run validation after n steps
valid_steps: 2000

# Default: 4000 - for large datasets, try up to 8000
warmup_steps: 4000
report_every: 100

# Number of GPUs, and IDs of GPUs
world_size: 1
gpu_ranks: [0]

# Batching
bucket_size: 262144
num_workers: 0  # Default: 2, set to 0 when RAM out of memory
batch_type: "tokens"
batch_size: 4096   # Tokens per batch, change when CUDA out of memory
valid_batch_size: 2048
max_generator_batches: 2
accum_count: [4]
accum_steps: [0]

# Optimization
model_dtype: "fp16"
optim: "adam"
learning_rate: 2
# warmup_steps: 8000
decay_method: "noam"
adam_beta2: 0.998
max_grad_norm: 0
label_smoothing: 0.1
param_init: 0
param_init_glorot: true
normalization: "tokens"
weight_decay: 0.0001

# Model
encoder_type: transformer
decoder_type: transformer
position_encoding: true
enc_layers: 6
dec_layers: 6
heads: 8
hidden_size: 512
word_vec_size: 512
transformer_ff: 2048
dropout_steps: [0]
dropout: [0.1]
attention_dropout: [0.1]
'''

with open("config.yaml", "w+") as config_yaml:
  config_yaml.write(config)

In [17]:
# Find the number of CPUs/cores on the machine
!nproc --all

16


In [18]:
# Build Vocabulary

# -config: path to your config.yaml file
# -n_sample: use -1 to build vocabulary on all the segment in the training dataset
# -num_threads: change it to match the number of CPUs to run it faster

!onmt_build_vocab -config config.yaml -n_sample -1 -num_threads 20


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.1.2 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "/venv/main/bin/onmt_build_vocab", line 5, in <module>
    from onmt.bin.build_vocab import main
  File "/venv/main/lib/python3.10/site-packages/onmt/__init__.py", line 2, in <module>
    import onmt.inputters
  File "/venv/main/lib/python3.10/site-packages/onmt/inputters/__init__.py", line 7, in <module>
    from onmt.inputters.text_utils import text_sort_key, process, numericalize, tensorify
  File "/venv/main/lib/python3.10/site-packages/onmt/inputters/text_utils.py", line 1, in <module>
    import torch
  File "

In [5]:
# Check if the GPU is active
!nvidia-smi -L

GPU 0: NVIDIA L40S (UUID: GPU-91006845-7b0b-d094-209b-a3a40a8ccfc6)


In [6]:
# Check if the GPU is visable to PyTorch
import torch

print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

gpu_memory = torch.cuda.mem_get_info(0)
print("Free GPU memory:", gpu_memory[0]/1024**2, "out of:", gpu_memory[1]/1024**2)


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.1.2 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "/usr/lib/python3.10/runpy.py", line 196, in _run_module_as_main
    return _run_code(code, main_globals, None,
  File "/usr/lib/python3.10/runpy.py", line 86, in _run_code
    exec(code, run_globals)
  File "/venv/main/lib/python3.10/site-packages/ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "/venv/main/lib/python3.10/site-packages/traitlets/config/application.py", line 1075, in launch_instance
    app.start()
  File "/venv/main/lib/python3.10/site-packages/ipykernel/kernelapp.p

True
NVIDIA L40S
Free GPU memory: 42824.5 out of: 45589.0625


In [19]:
# Train the NMT model
!onmt_train -config config.yaml


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.1.2 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "/venv/main/bin/onmt_train", line 5, in <module>
    from onmt.bin.train import main
  File "/venv/main/lib/python3.10/site-packages/onmt/__init__.py", line 2, in <module>
    import onmt.inputters
  File "/venv/main/lib/python3.10/site-packages/onmt/inputters/__init__.py", line 7, in <module>
    from onmt.inputters.text_utils import text_sort_key, process, numericalize, tensorify
  File "/venv/main/lib/python3.10/site-packages/onmt/inputters/text_utils.py", line 1, in <module>
    import torch
  File "/venv/main/l

## Perform Training on SoC Computer Cluster

1. **SSH to your SoC Computer Cluster and copy the train dev test files and config.yaml over**  

2. **Run `salloc` to acquire a GPU host:**  
   ```bash
   salloc -G nv -p gpu-long

3. **Find number of cores available:**  
   ```bash
   srun nproc --all

4. **Build vocabulary according to number of cores available:**  
   ```bash
   srun onmt_build_vocab -config config.yaml -n_sample -1 -num_threads 20

5. **Train it on `slurm` by setting time limit to one day:** 
    ```bash
    srun -t 1440 onmt_train -config config.yaml

## Translate

In [27]:
# Translate the "subworded" source file of the test dataset
# Change the model name, if needed.
!echo "---With WSD in test set---"
!onmt_translate -model models/model.wsd.pt -src en-zh.en-filtered-wsd-processed.en.subword.test -output zh.translated -gpu 0 -min_length 1

!echo "---Without WSD in test set---"
!onmt_translate -model models/model.wsd.pt -src en-zh.en-filtered-wsd.en.subword.test -output zh.translated.raw -gpu 0 -min_length 1

---With WSD in test set---

A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.1.2 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "/venv/main/bin/onmt_translate", line 5, in <module>
    from onmt.bin.translate import main
  File "/venv/main/lib/python3.10/site-packages/onmt/__init__.py", line 2, in <module>
    import onmt.inputters
  File "/venv/main/lib/python3.10/site-packages/onmt/inputters/__init__.py", line 7, in <module>
    from onmt.inputters.text_utils import text_sort_key, process, numericalize, tensorify
  File "/venv/main/lib/python3.10/site-packages/onmt/inputters/text_utils.py", line 1, in <module>
  

In [28]:
# Check the first 5 lines of the translation file
!echo "---With WSD in test set---"
!head -n 5 zh.translated

!echo "---Without WSD in test set---"
!head -n 5 zh.translated.raw

---With WSD in test set---
▁ 东 海岸 的 心脏 起 搏 器 , 德国 、 德国 、 德国 、 利 沙 尼亚 ▁ 利 沙 尼亚 、 拉 斯 维 亚 、 利 比 亚 ▁ 毛 里 斯 科 维 亚 、 马 里 兰 、 波 兰 、 波 兰 、 波 兰 ▁ 利 比 亚 , 现在 我 也可以 继续
▁我不知道 他们 打算 怎么 处理 这些东西 。
▁它 需要 一定 的 灵活 性 , ▁因为它 是 建立在 系统 的 要求 下 。
▁ 伟大的 老师 这样做 , ▁但是 伟大的 教育 也在 培养 皿 中 。
▁所以 , 我 只是 不停地 重复 这个 任务 , ▁最后 , 在 4 0 0 0 个 行 李 中 , ▁我 临 终 前 去 , 在 迷 宫 里 失 了 自己的 蛋白质 。
---Without WSD in test set---
▁ 内 科 研究所 , 德国 的 探险 家 , 克 美 沙 斯 洛 克 , 阿 桑 那 科 瓦 尼亚 , ▁ 德国 , 拉 斯 维 亚 , 利 比 亚 , 利 比 亚 , 利 比 亚 , 利 比 亚 , 利 比 亚 , 利 比 亚 , 利 桑 那 州 , 利 比 亚 , 利 桑 那 州 , ▁ 利 比 亚 , 阿尔 及 利亚 , 阿尔 及 利亚 , 阿 曼 利亚 , 利 比 亚 , 利 比 亚 , 还有 莫 尼亚 的 内 陆 , 现在 的 内 罗 德 黑 兰 及 利亚
▁我不知道 他们 要 怎么 处理 ,
▁ 无 意 中 有 无 意 无 动 于 衷 , 因为 这是一个 ▁ 系统 , 不 是个 自然 现象 。
▁ 慈善 家 会 做 这样的 事 , 但是 做 得很好 的 , ▁是 的 , 莱 斯特 们 , 莱 斯特 们 , 莱 斯特 们 。
▁ 就这样 , 我正在 不停地 翻 转 , ▁并 通过 这个 收集 化 身 的 化 身 , ▁这个 4 0 0 多 张 专辑 的 特 写 , ▁当我 接近 4 0 0 0 英尺 长 的 特 写 的时候 , ▁我 发现了 蛋白质 的 特 写 。


▁在 克 萨 斯 洛 伐 克 , 东 德国 , ▁ 欧 沙 尼亚 , 拉 维 亚 , 里 南 , ▁ 利 萨 维 亚 , 马 里 兰 , ▁ 奥 地 利 , 奥 地 利 , ▁ 奥 里 兰 , 塞 维 亚 , 我 还可以 继续 , ▁ 继续 努力 。
▁我不知道 他们 要 用 这些东西 做什么
▁它 需要 某种 交通 系统 来 规划 , ▁因为它 是一个 没有 交通 系统 的 系统 。
▁ 大 公司 会 这样做 , 但是 ▁ 大 公司 也 做 得很好 。
▁所以 , 我 只是 用 这种 排 骨 和 甘 地 的 薄 板 ▁ 插 到 最后 , 在 4 0 0 0 英里 的 地 上 , ▁当我 靠近 我的 神圣 门 时 , ▁我发现 蛋白质 。
---Without WSD in test set---
▁ 克 萨 斯 人 、 德国 人 、 巴西 人 、 ▁ 伊 沙 尼亚 人 、 拉 维 亚 人 、 利 桑 那 人 、 利 比 亚 人 、 ▁ 巴西 人 、 印度 流 浪 人 、 普 兰 人 、 ▁ 普 兰 人 、 巴西 人 、 巴西 人 、 塞 内 加 尔 、 塞 内 加 尔 和 内 加 尔 。
▁我不知道 他们 要 怎么 处理 这些东西 。
▁它 某种程度上 由 某种 系统 产生的 , ▁因为 这个系统 不是 自动 的 ,
▁ 霍 华 德 也 曾 这么做 , 但是 ▁ 霍 华 德 也是 , 也就是 奥 地 利 , 奥 斯 卡 , 等等 。
▁所以 , 我 试着 用 这种 甘 蔗 和 大 糖 ▁通过 这 根 甘 蓝 叉 ▁通过 这 4 0 0 0 个 苦 头 测试 ▁ 试着 粉 白色 , 找到 蛋白质


In [22]:
# If needed install/update sentencepiece
!pip3 install --upgrade -q sentencepiece

# Desubword the translation file
!python3 ./MT-Preparation/subwording/3-desubword.py ./target.model zh.translated 

!python3 ./MT-Preparation/subwording/3-desubword.py ./target.model zh.translated.raw

# Desubword test file
!python3 ./MT-Preparation/subwording/3-desubword.py ./target.model en-zh.zh-filtered-wsd.zh.subword.test


Done desubwording! Output: zh.salient.translated.desubword
Done desubwording! Output: zh.salient.translated.raw.desubword
Done desubwording! Output: en-zh.zh-filtered-wsd.zh.subword.test.desubword


In [24]:
# Desubword the target file (reference) of the test dataset
# Note: You might as well have split files *before* subwording during dataset preperation, 
# but sometimes datasets have tokeniztion issues, so this way you are sure the file is really untokenized.
!python3 ./MT-Preparation/subwording/3-desubword.py ./source.model en-zh.en-filtered-wsd-processed-salient.en.subword.test

Done desubwording! Output: en-zh.en-filtered-wsd-processed-salient.en.subword.test.desubword


In [25]:
# Check the first 5 lines of the desubworded translation file
!echo "---With WSD in test set---"
!head -n 5 zh.salient.translated.desubword

!echo "---Without WSD in test set---"
!head -n 5 zh.salient.translated.raw.desubword

# Check the first 5 lines of the desubworded reference
!echo "---Correct target---"
!head -n 5 en-zh.zh-filtered-wsd.zh.subword.test.desubword

---With WSD in test set---
东方的弗洛伦斯洛伐克、德国、希腊人、德国人、 拉塞维亚、利维亚、 里昂纳多利亚、 里维拉多利亚、 波兰、 波兰等各答的黑客都可以继续下去。
我不清楚他们会怎样处理这些废弃物
它多少有点概念,因为它是一种 不切实际的操作系统。
伟大的教师的确这样做了, 但是老师也采取了僵化的态度。
所以我就这么往下看, 最后,在加勒比海的这个400英尺处, 我失去全部的蛋白质。
---Without WSD in test set---
喀麦隆,埃及,德国, 德国,莱索托,莱索托,利比亚, 利桑尼亚, 利桑比亚, 利桑比亚, 利桑比亚, 利桑比亚, 利比亚,还有埃及,现在我可以去埃及。
他们将会做他们将要做的事情。
尼莱坞是反映系统的系统, 而不是机械化的。
虽说优秀的教师的工作是教师、 教师、教师、教师、记者、记者、记者、记者、记者,还是在教学中
使用Contunduston 和 4000 伏特的物品, 最后,当我努力去失去 4000 卡路里时,我找到蛋白质。
---Correct target---
在捷克斯洛伐克,东德 爱沙尼亚,拉脱维亚,立陶宛, 马里,马达加斯加, 波兰,菲律宾, 塞尔维亚,斯洛维尼亚的独裁政府,我可以继续, 还有现在的突尼斯和埃及。
我不知道他们将怎么处理那些东西。
这需要有探求精神 因为这整个系统不是以雕塑形式做成的
优秀的教师的确要这样做 但同时他们还会指导学生的学习 激发学生的兴趣,挑起学生的热情,赢得学生的关注
我按部就班地执行艰巨的任务 终于在第4000次筛选的时候 在我快发疯的时候 找到了符合标准的蛋白质


在克萨斯洛伐克,东德国, 欧沙尼亚,拉维亚,里南, 利萨维亚,马里兰, 奥地利,奥地利, 奥里兰,塞维亚,我还可以继续, 继续努力。
我不知道他们要用这些东西做什么
它需要某种交通系统来规划, 因为它是一个没有交通系统的系统。
大公司会这样做,但是 大公司也做得很好。
所以,我只是用这种排骨和甘地的薄板 插到最后,在4000英里的地上, 当我靠近我的神圣门时, 我发现蛋白质。
---Without WSD in test set---
克萨斯人、德国人、巴西人、 伊沙尼亚人、拉维亚人、利桑那人、利比亚人、 巴西人、印度流浪人、普兰人、 普兰人、巴西人、巴西人、塞内加尔、塞内加尔和内加尔。
我不知道他们要怎么处理这些东西。
它某种程度上由某种系统产生的, 因为这个系统不是自动的,
霍华德也曾这么做,但是 霍华德也是,也就是奥地利,奥斯卡,等等。
所以,我试着用这种甘蔗和大糖 通过这根甘蓝叉 通过这4000个苦头测试 试着粉白色,找到蛋白质
---Correct target---
在捷克斯洛伐克,东德 爱沙尼亚,拉脱维亚,立陶宛, 马里,马达加斯加, 波兰,菲律宾, 塞尔维亚,斯洛维尼亚的独裁政府,我可以继续, 还有现在的突尼斯和埃及。
我不知道他们将怎么处理那些东西。
这需要有探求精神 因为这整个系统不是以雕塑形式做成的
优秀的教师的确要这样做 但同时他们还会指导学生的学习 激发学生的兴趣,挑起学生的热情,赢得学生的关注
我按部就班地执行艰巨的任务 终于在第4000次筛选的时候 在我快发疯的时候 找到了符合标准的蛋白质


## Evaluation

In [13]:
# Download the BLEU script
!wget https://raw.githubusercontent.com/ymoslem/MT-Evaluation/main/BLEU/compute-bleu.py

--2025-04-13 07:00:40--  https://raw.githubusercontent.com/ymoslem/MT-Evaluation/main/BLEU/compute-bleu.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
200 OKequest sent, awaiting response... 
Length: 957 [text/plain]
Saving to: ‘compute-bleu.py.1’

compute-bleu.py.1   100%[===================>]     957  --.-KB/s    in 0s      

2025-04-13 07:00:40 (44.6 MB/s) - ‘compute-bleu.py.1’ saved [957/957]



In [14]:
# Install sacrebleu
!pip3 install sacrebleu

In [29]:
# Evaluate the translation (without subwording)
!echo "---With WSD in test set---"
!python3 compute-bleu.py en-zh.zh-filtered-wsd.zh.subword.test.desubword zh.translated.desubword

!echo "---Without WSD in test set---"
!python3 compute-bleu.py en-zh.zh-filtered-wsd.zh.subword.test.desubword zh.translated.raw.desubword

---With WSD in test set---
Reference 1st sentence: 在捷克斯洛伐克,东德 爱沙尼亚,拉脱维亚,立陶宛, 马里,马达加斯加, 波兰,菲律宾, 塞尔维亚,斯洛维尼亚的独裁政府,我可以继续, 还有现在的突尼斯和埃及。
MTed 1st sentence: 东海岸的心脏起搏器,德国、德国、德国、利沙尼亚 利沙尼亚、拉斯维亚、利比亚 毛里斯科维亚、马里兰、波兰、波兰、波兰 利比亚,现在我也可以继续
BLEU:  2.338224376063333
---Without WSD in test set---
Reference 1st sentence: 在捷克斯洛伐克,东德 爱沙尼亚,拉脱维亚,立陶宛, 马里,马达加斯加, 波兰,菲律宾, 塞尔维亚,斯洛维尼亚的独裁政府,我可以继续, 还有现在的突尼斯和埃及。
MTed 1st sentence: 内科研究所,德国的探险家,克美沙斯洛克,阿桑那科瓦尼亚, 德国,拉斯维亚,利比亚,利比亚,利比亚,利比亚,利比亚,利比亚,利桑那州,利比亚,利桑那州, 利比亚,阿尔及利亚,阿尔及利亚,阿曼利亚,利比亚,利比亚,还有莫尼亚的内陆,现在的内罗德黑兰及利亚
BLEU:  0.8075918460544266


Reference 1st sentence: 在捷克斯洛伐克,东德 爱沙尼亚,拉脱维亚,立陶宛, 马里,马达加斯加, 波兰,菲律宾, 塞尔维亚,斯洛维尼亚的独裁政府,我可以继续, 还有现在的突尼斯和埃及。
MTed 1st sentence: 在克萨斯洛伐克,东德国, 欧沙尼亚,拉维亚,里南, 利萨维亚,马里兰, 奥地利,奥地利, 奥里兰,塞维亚,我还可以继续, 继续努力。
BLEU:  2.3900016239366977
---Without WSD in test set---
Reference 1st sentence: 在捷克斯洛伐克,东德 爱沙尼亚,拉脱维亚,立陶宛, 马里,马达加斯加, 波兰,菲律宾, 塞尔维亚,斯洛维尼亚的独裁政府,我可以继续, 还有现在的突尼斯和埃及。
MTed 1st sentence: 克萨斯人、德国人、巴西人、 伊沙尼亚人、拉维亚人、利桑那人、利比亚人、 巴西人、印度流浪人、普兰人、 普兰人、巴西人、巴西人、塞内加尔、塞内加尔和内加尔。
BLEU:  1.5634516950417512
